# Running hofx 4D application in JEDI

This tutorial assumes that you have set up your work environment (which can be done by loading [Spack-Stack modules](https://spack-stack.readthedocs.io/en/latest/PreConfiguredSites.html) and that you have built [jedi-bundle](https://jointcenterforsatellitedataassimilation-jedi-docs.readthedocs-hosted.com/en/latest/using/building_and_running/building_jedi.html).

DISCOVER users can use [the jedi_bundle tool](https://github.com/geos-esm/jedi_bundle) to build JEDI on DISCOVER with the latest Spack Stack modules by following instructions [here](https://geos-esm.github.io/jedi_bundle/#/_platforms/building_jedi_code_on_discover_sles15).  


## prerequisite 
- `jedi-bundle` is already built in your `JEDI_BUILD`.
- Background files
- observation files
- Input geometry files


---

## Introducton:

In the HofX4D approach, we take a model state that is valid at the beginning of the assimilation window (blue circle). We also take all the observations measured within the assimilation window (red circles). Next, we must extract the model value at the observation time and location. Computing the model state at every observation time using the full non-linear model is costly. Instead, we use a simplified (linear) model (dark blue arrows) to move the model state forward in time using a user-specified time step (`tstep`) and create smaller windows. Observations within each smaller window are assumed to be valid at the beginning of the window. Next, for each smaller window, model values at the observation locations are extracted to compute the H(x) values. 

The JEDI executable for this application is `fv3jedi_hofx.x`

## HofX4D

<div style="text-align: center;">
    <img src="./figures/4DVar.jpg" width="500" height="300">
</div>

The HofX3D approach uses the model state (blue circle) valid at the middle of the assimilation window and assumes that all observations (red circles) within the assimialtion window are also valid at the same time. The model values at the observation locations are then spatially interpolated. Because computing the model state at each individual observation time would be computationally costly, we assume that all the observations (within the window) were measured at the same time as our model state, which is in the middle of the window.

In FV3-JEDI, the executable for the HofX3D application is `fv3jedi_hofx_nomodel.x`


### YAML structure (HofX3D):

To run `fv3jedi_hofx_nomodel.x`, JEDI requires in input file (in YAML format) which specifies the assimilation window, input geometry and state information, input observation information, and the observation operator. Below is an example to run the Column Retrieval observation operation with TEMPO observation and a low resolution GEOS-CF background. 

```yaml
# Beginning and length of assimilation window
time window:
  begin: '2023-08-05T15:00:00Z'
  length: PT6H
forecast length: PT6H

# Geometry of the state or background
geometry:
  fms initialization:
    namelist filename: inputs/geometry_input/fmsmpp.nml
  akbk: inputs/geometry_input/akbk72.nc4
  npx: 13
  npy: 13
  npz: 72
  field metadata override: inputs/geometry_input/geos.yaml

# Simplified linear model used to move the state forward in time
model:
  name: FV3LM
  namelist filename: inputs/geometry_input/input_geos_c12.nml
  tstep: PT15M
  lm_do_dyn: 1
  lm_do_trb: 0
  lm_do_mst: 0
  initialize model from A-Grid winds: true
  model variables:
  - air_pressure_thickness
  - volume_mixing_ratio_of_no2
  - volume_mixing_ratio_of_no
  - volume_mixing_ratio_of_o3
  - eastward_wind
  - northward_wind
  - air_temperature
  - geopotential_height_times_gravity_at_surface
  - water_vapor_mixing_ratio_wrt_moist_air

# Initial condition valid at the beginning of the window
initial condition:
  datetime: '2023-08-05T15:00:00Z'
  filetype: cube sphere history
  datapath: inputs/geos_cf_c12
  filename:  CF2.geoscf_jedi.c12.20230805T150000Z.nc4
  state variables:
  - air_pressure_thickness
  - volume_mixing_ratio_of_no2
  - volume_mixing_ratio_of_no
  - volume_mixing_ratio_of_o3
  - eastward_wind
  - northward_wind
  - air_temperature
  - geopotential_height_times_gravity_at_surface
  - water_vapor_mixing_ratio_wrt_moist_air

  field io names:
    air_pressure_thickness: DELP
    volume_mixing_ratio_of_no: 'NO'
    volume_mixing_ratio_of_no2: NO2
    volume_mixing_ratio_of_o3: O3
    air_pressure_at_surface: PS
    eastward_wind: ua
    northward_wind: va
    air_temperature: T
    geopotential_height_times_gravity_at_surface: phis
    water_vapor_mixing_ratio_wrt_moist_air: SPHU

# Observation and observation operator information
observations:
  observers:
  - obs space:
      name: tempo_no2_tropo
      obsdatain:
        engine:
          obsfile: inputs/obs/tempo_no2_tropo_20230805T150000Z.nc
          type: H5File
      obsdataout:
        engine:
          allow overwrite: true
          obsfile: output/fb.tempo_no2_tropo.20230805T150000Z.nc
          type: H5File
      observed variables:
      - nitrogendioxideColumn
      simulated variables:
      - nitrogendioxideColumn
    obs operator:
      name: ColumnRetrieval
      isApriori: false
      isAveragingKernel: true
      nlayers_retrieval: 72
      stretchVertices: topbottom
      tracer variables:
      - volume_mixing_ratio_of_no2

```

A few points about this experiment's YAML file:

- When comparing HofX3D and HofX4D YAMLs, note that the `state:` section is replaced by `model:` and `initial condition:` sections. 
- In this example, the `FV3LM` model is used as the "simplified model". This is a linearized version of the FV3 dynamical core. More information about fv3-jedi-linearmodel [here](https://jointcenterforsatellitedataassimilation-jedi-docs.readthedocs-hosted.com/en/7.0.0/inside/jedi-components/fv3-jedi/classes.html#tlm). 
- The initial condition is valid at the beginning of the window. 
- `tstep` is set to 15 minutes, meaning that there will be a model state available every 15 minutes. These states are computed by running the `FV3LM` model.
- Similar to HofX3D, observations are valid throughout the window.  

### Running HofX4D

First, you need to load the modules and set up the environment needed to run JEDI. More details available on [Spack-Stack docs](https://spack-stack.readthedocs.io/en/latest/PreConfiguredSites.html) and [JEDI docs](https://jointcenterforsatellitedataassimilation-jedi-docs.readthedocs-hosted.com/en/latest/using/building_and_running/building_jedi.html)

After loading the modules, set the path to your `JEDI_BUILD` and `MPIEXEC`. Running `which mpiexec` will return the path to your `mpiexec`. `JEDI_BUILD` is the path to your JEDI build directory. 

Note that for HofX4D we use `fv3jedi_hofx.x`



#### On Discover
On Discover, you can use Spack Stack Intel 1.9.0 and a a pre-build version of the JEDI-bundle: 

Load Spack Stack Intel 1.9 on Discover by executing:

`source /discover/nobackup/projects/gmao/advda/swell/jedi_modules/spackstack_1.9_intel`

Point to a pre-build version of the code here:
`/discover/nobackup/projects/jcsda/s2127/maryamao/geos-esm/jedi-work/build-intel-release/bin`


Here is an example of setting these two variables:

```bash
export MPIEXEC=/usr/local/intel/oneapi/2021/mpi/2021.10.0/bin/mpiexec
export JEDI_BUILD=/discover/nobackup/projects/jcsda/s2127/maryamao/geos-esm/jedi-work/build-intel-release/bin
```

All the files that you need are under `hofx` directory. So `cd hofx` and then you can run the application with this command: 


```bash
$MPIEXEC "-n" "6" $JEDI_BUILD/bin/fv3jedi_hofx.x hofx_fv3-geos_aero.yaml 2>&1 | tee log_hofx4d.txt
```

### examining the output of HofX3D run

Like the HofX3D exercise, you can view the output log and take note of the costly tasks. 
The output file has a similar format to the output file from the HofX3D exercise.

## Ctest example

You can use the ctest `fv3jedi_test_tier1_hofx_fv3-geos_cf` as a refence for the hofx4D application with geos_cf input files. In the `fv3-jedi` repository, in [`test/CMakeLists.txt`](https://github.com/JCSDA/fv3-jedi/blob/develop/test/CMakeLists.txt) look for `fv3jedi_test_tier1_hofx_fv3-geos_cf` test:

```yaml
ecbuild_add_test( TARGET   fv3jedi_test_tier1_hofx_fv3-geos_cf
                  MPI      6
                  ARGS     testinput/hofx_fv3-geos_cf.yaml
                  COMMAND  fv3jedi_hofx.x )
```

Here you can see the input yaml [`testinput/hofx_fv3-geos_cf.yaml`](https://github.com/JCSDA-internal/fv3-jedi/blob/develop/test/testinput/hofx_fv3-geos_cf.yaml) and the executable `fv3jedi_hofx.x` which is used to run the test.
